## Load test VLM notebook
This notebook will perform a quick load test on the endpoint deployed for a image model

In [0]:
%pip install mlflow
dbutils.library.restartPython()

In [0]:
import requests
import pandas as pd
import numpy as np
from PIL import Image
import io
import base64
import mlflow
from mlflow.models.utils import convert_input_example_to_serving_input
from mlflow.utils.databricks_utils import get_databricks_env_vars
import json
import time

In [0]:
def reduce_image_size(img, factor=4):
    width, height = img.size
    new_size = (width // factor, height // factor)
    return img.resize(new_size)

def pillow_image_to_base64_string(img):
    buffered = io.BytesIO()
    img.save(buffered, format="JPEG")
    return base64.b64encode(buffered.getvalue()).decode("utf-8")

# example_image_url = "http://images.cocodataset.org/val2017/000000039769.jpg"
# example_image = Image.open(requests.get(example_image_url, stream=True).raw)

example_image_url = "/Volumes/uc_sriharsha_jana/test_db/shjdata/test_image.png"
example_image = Image.open(example_image_url).convert("RGB")

example_image_resized = reduce_image_size(example_image)
example_image_base64 = pillow_image_to_base64_string(example_image_resized)
input_example = pd.DataFrame().from_records([{"user_prompt": "describe the image and note the objects in the image", "image": example_image_base64}])

In [0]:
example_image_resized

In [0]:
input_dict = dict()
input_dict["dataframe_records"] = input_example.to_dict("records")
input_dict["params"] = {"temperature": 0.1, "max_new_tokens": 512, "top_p": 0.95}

In [0]:
# DATA = convert_input_example_to_serving_input(input_example)
DATA = json.dumps(input_dict)

DATABRICKS_TOKEN = "eyJraWQiOiJkZmJjOWVmMThjZTQ2ZTlhMDg2NWZmYzlkODkxYzJmMjg2NmFjMDM3MWZiNDlmOTdhMDg1MzBjNWYyODU3ZTg4IiwidHlwIjoiYXQrand0IiwiYWxnIjoiUlMyNTYifQ.eyJjbGllbnRfaWQiOiJkYXRhYnJpY2tzLXNlc3Npb24iLCJzY29wZSI6ImFsbC1hcGlzIiwiYXV0aG9yaXphdGlvbl9kZXRhaWxzIjpbeyJvYmplY3RfdHlwZSI6InNlcnZpbmctZW5kcG9pbnRzIiwib2JqZWN0X3BhdGgiOiIvc2VydmluZy1lbmRwb2ludHMvNDg0MTBhMDQ2MzljNDIxMmIxOTY4YTkxMWU0YTFmZmQiLCJhY3Rpb25zIjpbInF1ZXJ5X2luZmVyZW5jZV9lbmRwb2ludCJdLCJhbm5vdGF0aW9ucyI6eyJlbmRwb2ludE5hbWUiOiJzaGotbGxhbWEtdmxtIn0sInR5cGUiOiJ3b3Jrc3BhY2VfcGVybWlzc2lvbiJ9XSwiaXNzIjoiaHR0cHM6Ly9lMi1kZW1vLWZpZWxkLWVuZy5jbG91ZC5kYXRhYnJpY2tzLmNvbS9vaWRjIiwiYXVkIjoiMTQ0NDgyODMwNTgxMDQ4NSIsInN1YiI6InNyaWhhcnNoYS5qYW5hQGRhdGFicmlja3MuY29tIiwiaWF0IjoxNzM5MzUwMDU3LCJleHAiOjE3MzkzNTM2NTcsImp0aSI6IjFlNjFkZDdiLTQ3MjktNDFhYy04ODFjLTBhMDY2MjUxNjAwZSJ9.cWcnJ7aWppI7-GPo6C4104PkXAakZ-aUNERoqWISDOaZQMxCn7soin2Y2vj2jUeqMCzFfF6KjohToH__IzpSFbrh0EsuBlhe_ytI_nC7OboUBgvqoGn8EGV0IhMypOCvm7E3RfkfERzjjoz5-kwPYbryfcyO32NYs3Bomj5B8rgrtwn-vF73BQIoV2VpgPI6IUJ3uKI8R9U7V7zpXWT0_0H8xSTDdCcGk2DtETJfPEqRik-wqkyBTdsJr4Y9UyBy8iMzayckAHUxcpgz4jEXAxQ_ukHIjGlzFJVec9-iBQ40jGwhUcPlwtqYTq3I5Sr_NqLx6GSMIxMwfkF_lzMdsg"

ENDPOINT = "https://48410a04639c4212b1968a911e4a1ffd.serving.cloud.databricks.com/1444828305810485/serving-endpoints/shj-llama-vlm/invocations"

In [0]:
session = requests.Session()
req = requests.Request('POST', ENDPOINT, headers={'Authorization': f'Bearer {DATABRICKS_TOKEN}', 'Content-Type': 'application/json'}, data=DATA)
prepped = req.prepare()

In [0]:
num_req = 2

print(f"Sending {num_req} requests to the endpoint as a sanity check.\n")

has_failed_request = False
for i in range(num_req):
  resp = session.send(prepped)
  if resp.status_code != 200:
    has_failed_request = True
    print("Request failed. Please DO NOT move to the next step. Check if the token/endpoint/data are all valid.")
    print("  response code: " + str(resp.status_code))
    print("  response body: " + str(resp.text))
    break
  if i == 0:
    print("First request succeeded, please sanity-check the response: " + str(resp.text))
  print(f"Request succedded {i}")
if not has_failed_request:
  print(f"\nCongratulations! All {num_req} requests succeeded. Please proceed to the next step.")

In [0]:
num_req = 30

print("Sending %d requests to measure the latency..." % num_req)
latencies_seconds = []
for i in range(num_req):
  start = time.perf_counter()
  resp = session.send(prepped)
  end = time.perf_counter()
  latencies_seconds.append((end - start))

In [0]:
latency_p50 = np.percentile(latencies_seconds, 50)
latency_p90 = np.percentile(latencies_seconds, 90)

print("The 50th percentile latency is %d seconds and the 90th percentile latency is %d seconds for total %d requests." % (latency_p50, latency_p90, num_req))

In [0]:
latency_p50 = np.percentile(latencies_seconds, 50)
latency_p90 = np.percentile(latencies_seconds, 90)

print("The 50th percentile latency is %d seconds and the 90th percentile latency is %d seconds for total %d requests." % (latency_p50, latency_p90, num_req))

In [0]:
latency_p50 = np.percentile(latencies_seconds, 50)
latency_p90 = np.percentile(latencies_seconds, 90)

print("The 50th percentile latency is %d seconds and the 90th percentile latency is %d seconds for total %d requests." % (latency_p50, latency_p90, num_req))